# Monitoring a deployed system for drift: PPIMeanMonitor

**`PPIMeanMonitor`** watches a quality metric across successive batches of production data and raises an alarm the moment there is statistically valid evidence that it has drifted past a threshold you fix in advance. This is necessary because a deployed system's quality is rarely static: traffic shifts and upstream models change, so a metric that looked fine at launch can drift silently, and a single-shot estimate made at first deployment time cannot catch that.

---

**What you will learn:**

- Why re-checking a fresh confidence interval multiple times inflates the false-alarm rate
- How `PPIMeanMonitor` gives an anytime-valid alternative that stays safe under repeated checks
- How to read its running-mean curve, its confidence bound, and its alarms

## The problem: the metric can drift after launch, and repeated checking breaks validity

Your customer-facing assistant handles an open-ended troubleshooting flow, and hallucinations are a real risk there. Every week, a fresh batch of conversations is judged by your LLM judge, and a handful are sent for human review. Your team has fixed a business threshold on the hallucination rate, and wants to know as soon as the rate has drifted past it.

The tempting approach is to compute a fresh confidence interval every week and raise an alarm the moment one crosses the threshold. This is **peeking**: each weekly test carries its own false-alarm probability, and checking after every batch compounds those chances, so a false alarm becomes likely over a long enough monitoring horizon even if nothing has actually drifted. The [Monitors user guide](../user_guide/monitors/) derives why this happens and how `PPIMeanMonitor` avoids it, an anytime-valid confidence sequence which stays safe no matter how often you look. This tutorial focuses on what that means for you in practice.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import PPIMeanEstimator
from glide.monitors import PPIMeanMonitor
from glide.samplers import StratifiedSampler
from glide.simulators import generate_stratified_binary_dataset, simulate_annotation

# ── Colour palette ──────────────────────────────────────────
C_ESTIMATE = "#95A5A6"  # per-batch point estimate — grey
C_NAIVE = "#E74C3C"  # naive baseline alarm        — red-orange
C_MONITOR = "#27AE60"  # monitor running mean      — green
C_ALARM = "#8E44AD"  # monitor alarm               — purple
C_THRESHOLD = "#2C3E50"  # threshold           — dark slate

# ── Global plot style ───────────────────────────────────────
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "font.size": 18,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "legend.fontsize": 14,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.titlesize": 19,
    }
)

## Simulating a batched production stream

We use `generate_stratified_binary_dataset` to simulate one batch of conversations per week, each with its own true hallucination rate, proxy rate, and correlation, stacked into a single stream with an integer identifying the chronological batch each row belongs to.

`simulate_batches` below builds such a stream from a list of per-week true hallucination rates. Every week gets `BATCH_SIZE` proxy-judged conversations, of which `LABELS_PER_BATCH` are sent for human review: `StratifiedSampler` allocates that review budget across weeks, and the human labels are revealed only for selected conversations, leaving the rest as `np.nan`. We fix a threshold above the stationary rate, so a stable week is comfortably compliant and a real regression clearly is not. All batches in the stream are monitored.

In [ ]:
N_BATCHES = 60
BATCH_SIZE = 100
LABELS_PER_BATCH = 20
STATIONARY_MEAN = 0.30
PROXY_BIAS = 0.1
CORRELATION = 0.7
THRESHOLD = 0.35


def simulate_batches(true_means, random_seed):
    proxy_means = true_means + PROXY_BIAS
    correlations = np.full(len(true_means), CORRELATION)
    y_true_oracle, y_proxy, batches = generate_stratified_binary_dataset(
        n_total=np.full(len(true_means), BATCH_SIZE),
        true_mean=true_means,
        proxy_mean=proxy_means,
        correlation=correlations,
        random_seed=random_seed,
    )
    xi = StratifiedSampler().sample(
        y_proxy,
        groups=batches,
        n_samples=LABELS_PER_BATCH * len(true_means),
        strategy="proportional",
        random_seed=random_seed,
    )
    y_true = simulate_annotation(y_true_oracle, xi)
    return y_true, y_proxy, batches


true_means_no_drift = np.full(N_BATCHES, STATIONARY_MEAN)
y_true_no_drift, y_proxy_no_drift, batches_no_drift = simulate_batches(true_means_no_drift, random_seed=7)

n_labeled = np.sum(~np.isnan(y_true_no_drift))
print(f"Total conversations : {len(y_true_no_drift):,}")
print(f"  Human-reviewed     : {n_labeled:,}")
print(f"  Number of weeks    : {len(np.unique(batches_no_drift))}")

## The naive baseline: one confidence interval per batch

The tempting-but-invalid approach simply calls `PPIMeanEstimator` once per week, comparing that week's confidence interval to the threshold and alarming whenever the interval lies above it. 

In [ ]:
def naive_per_batch_alarms(y_true, y_proxy, batches, threshold, n_batches):
    means = np.zeros(n_batches)
    lower_bounds = np.zeros(n_batches)
    alarms = np.zeros(n_batches, dtype=bool)
    for batch_id in range(n_batches):
        batch_mask = batches == batch_id
        batch_result = PPIMeanEstimator().estimate(y_true[batch_mask], y_proxy[batch_mask])
        means[batch_id] = batch_result.mean
        lower_bounds[batch_id] = batch_result.confidence_interval.lower_bound
        alarms[batch_id] = batch_result.confidence_interval.lower_bound > threshold
    return means, lower_bounds, alarms

In [ ]:
def plot_monitor_vs_baseline(batch_means, monitor_result, naive_alarms, title):
    weeks = np.arange(1, len(batch_means) + 1)
    fig, ax = plt.subplots(figsize=(10, 5.5))

    ax.scatter(weeks, batch_means, color=C_ESTIMATE, s=28, alpha=0.8, zorder=3, label="Per-batch estimate")
    ax.plot(weeks, monitor_result.running_means, color=C_MONITOR, linewidth=2.5, zorder=4, label="Monitor running mean")
    ax.axhline(THRESHOLD, color=C_THRESHOLD, linestyle="--", linewidth=1.8, zorder=2, label="Threshold")

    if naive_alarms.any():
        ax.scatter(
            weeks[naive_alarms],
            batch_means[naive_alarms],
            marker="x",
            s=110,
            linewidths=2.5,
            color=C_NAIVE,
            zorder=5,
            label="Naive baseline alarm",
        )

    if monitor_result.drift_detected:
        alarm_week = monitor_result.first_alarm_index + 1
        alarm_label = f"Monitor alarm (week {alarm_week})"
        ax.axvline(alarm_week, color=C_ALARM, linestyle=":", linewidth=2.2, zorder=2, label=alarm_label)

    ax.set_xlabel("Week")
    ax.set_ylabel("Hallucination Rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper left", framealpha=0.9)
    plt.tight_layout()
    plt.show()

## Scenario 1: no drift, no false alarm

The stream is fully stationary: the true hallucination rate stays below the threshold. Neither method should alarm, except by chance.

In [ ]:
means_no_drift, lower_no_drift, naive_alarms_no_drift = naive_per_batch_alarms(
    y_true_no_drift, y_proxy_no_drift, batches_no_drift, THRESHOLD, N_BATCHES
)
monitor_result_no_drift = PPIMeanMonitor().detect(
    y_true_no_drift,
    y_proxy_no_drift,
    batches_no_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    metric_name="Hallucination Rate",
)
naive_alarm_weeks = np.arange(1, N_BATCHES + 1)[naive_alarms_no_drift]

print(f"Naive baseline alarms at week(s): {naive_alarm_weeks}")
print(f"Monitor drift detected: {monitor_result_no_drift.drift_detected}")
print(
    f"Monitor running mean / bound after 60 weeks: {monitor_result_no_drift.running_means[-1]:.1%} / "
    f"{monitor_result_no_drift.confidence_bounds[-1]:.1%}"
)

In [ ]:
plot_monitor_vs_baseline(
    means_no_drift,
    monitor_result_no_drift,
    naive_alarms_no_drift,
    title="Scenario 1: stationary stream (no drift)",
)

At some point, ordinary sampling noise pushes one week's confidence interval above the threshold, and the naive baseline raises a false alarm, purely from **peeking**: with a fresh, independent-looking test every week, the chance of at least one false alarm climbs the longer you keep checking, even though the hallucination rate never moved. `PPIMeanMonitor`, in contrast, never alarms across the full stream: its running mean stays below the threshold, with an anytime-valid lower bound safely underneath it as well.

## Scenario 2: slow drift, early alarm

Now a change progressively regresses quality: partway through the stream, the hallucination rate ramps up gradually over several weeks and then stays elevated for the rest of the deployment.

In [ ]:
DRIFT_START = 10
DRIFT = 0.35
RAMP_LENGTH = 8
ramp = np.linspace(STATIONARY_MEAN, STATIONARY_MEAN + DRIFT, RAMP_LENGTH)
true_means_slow_drift = np.hstack(
    [
        np.full(DRIFT_START, STATIONARY_MEAN),
        ramp,
        np.full(N_BATCHES - DRIFT_START - RAMP_LENGTH, STATIONARY_MEAN + DRIFT),
    ]
)
y_true_slow_drift, y_proxy_slow_drift, batches_slow_drift = simulate_batches(true_means_slow_drift, random_seed=0)

means_slow_drift, lower_slow_drift, naive_alarms_slow_drift = naive_per_batch_alarms(
    y_true_slow_drift, y_proxy_slow_drift, batches_slow_drift, THRESHOLD, N_BATCHES
)
monitor_result_slow_drift = PPIMeanMonitor().detect(
    y_true_slow_drift,
    y_proxy_slow_drift,
    batches_slow_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    metric_name="Hallucination Rate",
)

naive_alarm_weeks = np.flatnonzero(naive_alarms_slow_drift) + 1
print(f"Naive baseline alarms: {len(naive_alarm_weeks)} weeks, first at week {naive_alarm_weeks[0]}")
alarm_index = monitor_result_slow_drift.first_alarm_index
if alarm_index is not None:
    print(f"Monitor drift detected: True, first alarm at week {alarm_index + 1}")

In [ ]:
plot_monitor_vs_baseline(
    means_slow_drift,
    monitor_result_slow_drift,
    naive_alarms_slow_drift,
    title="Scenario 2: slow drift, then a stable regression",
)

Once the rate has settled at its new, higher level, most weekly estimates land far enough above the threshold that the naive baseline starts alarming too, and keeps alarming for most of the rest of the stream. In hindsight, many of those alarms are correct, the rate genuinely is above the threshold by then, but that is only visible after the fact. In real time, a team watching the naive baseline has no way to tell a validated alarm from a lucky (or unlucky) one, because its error rate was never controlled for repeated testing in the first place: Scenario 1 already showed it firing on a completely stationary stream.

`PPIMeanMonitor`'s single alarm carries no such ambiguity: it is backed by the same anytime-valid guarantee demonstrated in Scenario 1, so under no drift the chance it would ever fire at all across the whole monitoring horizon is at most `1 - confidence_level`. At the moment it fires, its anytime-valid bound has just crossed the threshold. By the end of the stream, the running mean and its bound both sit comfortably clear of the threshold.

## Reading the result object

Both scenarios returned a `PredictionPoweredMeanMonitoringResult`. Its public surface covers everything used above, plus a formatted summary.

In [ ]:
print(f"Drift detected      : {monitor_result_slow_drift.drift_detected}")
print(f"First alarm index    : {monitor_result_slow_drift.first_alarm_index}")
print(f"Alarm threshold      : {monitor_result_slow_drift.alarm_threshold}")
print(f"Running means (last 3): {np.round(monitor_result_slow_drift.running_means[-3:], 3)}")
print(f"Confidence bounds (last 3): {np.round(monitor_result_slow_drift.confidence_bounds[-3:], 3)}")
print()
print(monitor_result_slow_drift)

## How to size and schedule your batches

Looking back at the two figures above, four plain-language rules govern how to split your annotation budget into batches:

1. **Give every batch enough human labels that its own estimate is not pure noise.** `PPIMeanMonitor` requires at least 2 labeled and 2 unlabeled samples per batch to run at all, but for a rate-like metric such as this hallucination rate, tens of labels per batch is a safer target. The number of labels per week used in this tutorial's demo is deliberately small, which is exactly why the grey per-batch points in both figures are so jagged.
2. **Keep batch sizes roughly comparable.** Every batch counts equally in the running mean, so one unusually tiny batch injects a noisy estimate at full weight, the same way an outlier week would skew a plain average.
3. **Prefer more, smaller batches over fewer, bigger ones, on the same natural cadence.** Detection speed improves with the number of batches, while the precision of any one estimate depends on its own label count. Schedule one batch per natural review period, weekly here, rather than merging several weeks together: this also keeps the running mean responsive to recent drift, as noted in the [Monitors user guide](../user_guide/monitors/). The one exception is merging adjacent batches that are each individually too small for a stable estimate.
4. **Do not expect an alarm in the first few batches.** The anytime-valid bound starts wide and tightens as batches accumulate, by design, so it complements rather than replaces eyeballing your dashboard during the first few weeks after a launch.

## Summary

| | Per-batch baseline | `PPIMeanMonitor` |
|-|---------------------|-------------------|
| Valid under repeated looks? | No: its false-alarm rate is uncontrolled and grows the longer you keep checking | Yes: bounded by `1 - confidence_level` over the whole monitoring horizon |
| Detects a real, sustained drift? | Eventually, but with no guarantee on when or how many of its alarms are trustworthy | Yes, with a single alarm backed by the anytime-valid guarantee |
| What it needs | Nothing beyond `PPIMeanEstimator`, called on the wrong cadence | `PPIMeanMonitor`, built for exactly this repeated-testing setting |

**Key takeaways:**

1. **Repeated significance testing is not a monitor.** Scenario 1 showed the per-batch baseline raise a false alarm on a perfectly stationary stream, purely from checking every week.
2. **An anytime-valid confidence sequence makes checking every week safe.** `PPIMeanMonitor`'s guarantee holds simultaneously across all looks, so its single alarm in Scenario 2 can be trusted the moment it fires, without waiting to see whether later weeks confirm it.
3. **The naive baseline's alarms carry no error-rate guarantee, correct or not.** Scenario 2's naive alarms happened to line up with a real regression, but Scenario 1 already showed the same procedure crying wolf under no drift at all: there is no way to tell the two situations apart from the naive alarms alone.

---

*Want to go further? The [PPRM scientific validation notebook](../../deep_dive/scientific_validation/monitors/ppi/) provides rigorous empirical evidence on false-alarm control and detection power, and the [Monitors user guide](../user_guide/monitors/) derives the anytime-valid confidence sequence used here from first principles.*